In [2]:
!wget -O shaketext.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-03-20 01:44:13--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘shaketext.txt’

shaketext.txt       100%[===================>]   1.06M  --.-KB/s    in 0.09s   

2026-03-20 01:44:13 (11.8 MB/s) - ‘shaketext.txt’ saved [1115394/1115394]



In [3]:
# decode text to utf-8 for reading
with open('shaketext.txt', 'r', encoding='utf-8') as f:
  text = f.read()


In [4]:
print('length: ', len(text))

length:  1115394


In [5]:
chars = list(set(text))
vocab_size = len(chars)
print(chars)
print(''.join(chars))
print(len(chars))

['O', 'L', ':', 'W', 'N', 'V', 'w', 'K', 'd', 'm', 'T', 'B', 'F', 'q', '3', 'D', 'l', 'j', 'Z', 'Y', 'Q', ' ', 'n', 'f', 'M', 'a', '$', 'x', 'p', 't', 's', 'P', 'I', 'G', 'r', ';', 'A', 'C', 'h', 'E', 'X', 'b', '\n', 'c', ',', 'R', '!', 'o', '&', 'v', '?', 'u', 'H', 'z', 'e', 'S', 'U', 'J', 'k', "'", 'i', 'y', '.', 'g', '-']
OL:WNVwKdmTBFq3DljZYQ nfMa$xptsPIGr;AChEXb
c,R!o&v?uHzeSUJk'iy.g-
65


In [6]:
stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda i: ''.join(itos[e] for e in i)

print(stoi)
print(encode("CtxN"))
print(decode(encode("CtxN")))

{'O': 0, 'L': 1, ':': 2, 'W': 3, 'N': 4, 'V': 5, 'w': 6, 'K': 7, 'd': 8, 'm': 9, 'T': 10, 'B': 11, 'F': 12, 'q': 13, '3': 14, 'D': 15, 'l': 16, 'j': 17, 'Z': 18, 'Y': 19, 'Q': 20, ' ': 21, 'n': 22, 'f': 23, 'M': 24, 'a': 25, '$': 26, 'x': 27, 'p': 28, 't': 29, 's': 30, 'P': 31, 'I': 32, 'G': 33, 'r': 34, ';': 35, 'A': 36, 'C': 37, 'h': 38, 'E': 39, 'X': 40, 'b': 41, '\n': 42, 'c': 43, ',': 44, 'R': 45, '!': 46, 'o': 47, '&': 48, 'v': 49, '?': 50, 'u': 51, 'H': 52, 'z': 53, 'e': 54, 'S': 55, 'U': 56, 'J': 57, 'k': 58, "'": 59, 'i': 60, 'y': 61, '.': 62, 'g': 63, '-': 64}
[37, 29, 27, 4]
CtxN


In [7]:
import torch
encodedData = torch.tensor(encode(text), dtype=torch.long)
print(encodedData.shape, encodedData.dtype)
print(encodedData[:10])

torch.Size([1115394]) torch.int64
tensor([12, 60, 34, 30, 29, 21, 37, 60, 29, 60])


In [8]:
train_length = int(0.9*len(encodedData))
train_data = encodedData[:train_length]
val_data = encodedData[train_length:]
print(len(train_data))
print(len(val_data))

1003854
111540


In [9]:
block_size = 8
print(train_data[:block_size+1])

tensor([12, 60, 34, 30, 29, 21, 37, 60, 29])


In [10]:
x = train_data[:block_size]
y = train_data[1:block_size + 1]
for t in range(block_size):
  source = x[:t+1]
  target = y[t]
  print("When input is", source, "then target is ",target)

When input is tensor([12]) then target is  tensor(60)
When input is tensor([12, 60]) then target is  tensor(34)
When input is tensor([12, 60, 34]) then target is  tensor(30)
When input is tensor([12, 60, 34, 30]) then target is  tensor(29)
When input is tensor([12, 60, 34, 30, 29]) then target is  tensor(21)
When input is tensor([12, 60, 34, 30, 29, 21]) then target is  tensor(37)
When input is tensor([12, 60, 34, 30, 29, 21, 37]) then target is  tensor(60)
When input is tensor([12, 60, 34, 30, 29, 21, 37, 60]) then target is  tensor(29)


In [11]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    get_block = lambda start: data[start: start+block_size]
    x = torch.stack([get_block(i) for i in ix])
    y = torch.stack([get_block(i+1) for i in ix])
    return x, y

xb, yb = get_batch('train')
print('stack x:')
print(xb.shape)
print(xb)
print('stack y:')
print(yb.shape)
print(yb)

print("We have total 4 sequences")
for b in range(batch_size):
  for t in range(block_size):
    context = xb[b, :t+1]
    target = yb[b,t]
    print("When input is", context, "then target is ",target)



stack x:
torch.Size([4, 8])
tensor([[ 1, 54, 29, 59, 30, 21, 38, 54],
        [23, 47, 34, 21, 29, 38, 25, 29],
        [22, 29, 21, 29, 38, 25, 29, 21],
        [24, 39,  0,  2, 42, 32, 21, 28]])
stack y:
torch.Size([4, 8])
tensor([[54, 29, 59, 30, 21, 38, 54, 25],
        [47, 34, 21, 29, 38, 25, 29, 21],
        [29, 21, 29, 38, 25, 29, 21, 38],
        [39,  0,  2, 42, 32, 21, 28, 25]])
We have total 4 sequences
When input is tensor([1]) then target is  tensor(54)
When input is tensor([ 1, 54]) then target is  tensor(29)
When input is tensor([ 1, 54, 29]) then target is  tensor(59)
When input is tensor([ 1, 54, 29, 59]) then target is  tensor(30)
When input is tensor([ 1, 54, 29, 59, 30]) then target is  tensor(21)
When input is tensor([ 1, 54, 29, 59, 30, 21]) then target is  tensor(38)
When input is tensor([ 1, 54, 29, 59, 30, 21, 38]) then target is  tensor(54)
When input is tensor([ 1, 54, 29, 59, 30, 21, 38, 54]) then target is  tensor(25)
When input is tensor([23]) then targe

In [12]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BiagramLanguageModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    print(self.token_embedding_table)

  def forward(self, idx, targets=None):
    logits = self.token_embedding_table(idx)

    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, idx, max_new_tokens):
    # generate with max_new_tokens elements append into idx
    for _ in range(max_new_tokens):
      logits, loss = self(idx)
      logits = logits[:, -1, :]
      probs = F.softmax(logits, dim=-1)
      idx_next = torch.multinomial(probs, num_samples=1)
      idx = torch.cat((idx, idx_next), dim=1)
    return idx


m = BiagramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss) # should be -ln(1/65)

idx = torch.zeros((1,1), dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))




Embedding(65, 65)
torch.Size([32, 65])
tensor(5.1028, grad_fn=<NllLossBackward0>)
OPf bMIBqbjMazIXihiLDFHSKtjmmTDnSvxfQKSBT?E:zogyvQ&RD!-XtF':mXM RyciugZnYGYeOyHC;uH.f;;ciKklSAjUrfMRn


In [13]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [14]:
batch_size = 32
for steps in range(10000):
  # 100: 4.x
  # 1000: 3.x

    xb, yb = get_batch('train')

    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())


2.3799660205841064


In [15]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))

O?
INo drdifuern'TRSher Pre.
D: wonouto' th
KII y-wive thth lo te st call pe feacre. w:
f t ack 'sencowen I
berey,

ABus, ty we d r it w,
Lie:
SThntathot ichishindend Bupot in ve uksto nt t ouge
A:
Lins:
S y s altotitollad---'s, arar wipes
Wheecetel w O:
C:
I HERESlind t, sif bathobu malean shen abeners?
COngh:
Agitece m ar thy o tit blo but too O:
TEO: he atabl nanthit my.
Yous ce hitorsivirst frnos
STh ancul '

Ag o wh paverasoust!
Therysesthe he.

Bur me tande'?

Bur f s mpe w theco wsousor l;


In [16]:
torch.manual_seed(1337)
B,T,C = 4,8,2
x = torch.randn(B,T,C)
x
x.shape

torch.Size([4, 8, 2])

In [17]:
# x[b,T] = sum(x[b,i] for i in range (0,T))
xbow = torch.zeros((B,T,C))
for b in range (B):
  for t in range(T):
    xprev = x[b, :t+1]
    xbow[b,t] = torch.mean(xprev, 0)
xbow

tensor([[[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]],

        [[ 1.3488, -0.1396],
         [ 0.8173,  0.4127],
         [-0.1342,  0.4395],
         [ 0.2711,  0.4774],
         [ 0.2421,  0.0694],
         [ 0.0084,  0.0020],
         [ 0.0712, -0.1128],
         [ 0.2527,  0.2149]],

        [[-0.6631, -0.2513],
         [ 0.1735, -0.0649],
         [ 0.1685,  0.3348],
         [-0.1621,  0.1765],
         [-0.2312, -0.0436],
         [-0.1015, -0.2855],
         [-0.2593, -0.1630],
         [-0.3015, -0.2293]],

        [[ 1.6455, -0.8030],
         [ 1.4985, -0.5395],
         [ 0.4954,  0.3420],
         [ 1.0623, -0.1802],
         [ 1.1401, -0.4462],
         [ 1.0870, -0.4071],
         [ 1.0430, -0.1299],
         [ 1.1138, -0.1641]]])

In [18]:
# using matrix multiply to produce same result
wei = torch.tril(torch.ones(T,T))
wei = wei / wei.sum(1, keepdim=True)
print(wei)
xbow2 = wei @ x
print("xbow2: ", xbow2)
print("xbow: ",xbow)
diff = (xbow - xbow2).abs()
print("Max difference:", diff.max().item())
print("Mean difference:", diff.mean().item())
print(torch.allclose(xbow, xbow2, atol=1e-7))

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])
xbow2:  tensor([[[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]],

        [[ 1.3488, -0.1396],
         [ 0.8173,  0.4127],
         [-0.1342,  0.4395],
         [ 0.2711,  0.4774],
         [ 0.2421,  0.0694],
         [ 0.0084,  

In [19]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
print(tril)
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
print(wei)

xbow3 = wei @ x
torch.allclose(xbow, xbow3, atol=1e-7)

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])


True

# Scaled dot-product attention

# $$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In [31]:
torch.manual_seed(1337)
B, T, C = 4, 8, 32   # 4 batches, 8 tokens per sequence, 32 channels (embedding dim)
x = torch.randn(B, T, C)

head_size = 16  # size of Q/K/V space (smaller than C for efficiency)

# 3 linear projections: map each token from C=32 → head_size=16
# bias=False: simpler, standard in attention
key   = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)    # (B, T, 16): "what does each token have?"
q = query(x)  # (B, T, 16): "what is each token looking for?"

# Q @ Kᵀ: dot product between every pair of tokens → affinity score
# (B,T,16) @ (B,16,T) → (B,T,T)
# wei[b,i,j] = how much token i should attend to token j
wei = q @ k.transpose(-2, -1)

# Multiply with 1/căn(head_size) to prevent too weak , caused the large dot products -> softmax = one-hot
# Without this: large dot products → softmax ≈ one-hot → vanishing gradients
wei = wei * head_size**-0.5

# Causal mask: token at position t can only see positions 0..t (no future)
tril = torch.tril(torch.ones(T, T))
print(wei[0])
# wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))  # future → -inf -> current elements only predict based on previous, not on the future
print(wei[0])
wei = F.softmax(wei, dim=-1)  # Normalize distribution -> sum row equals to 1 and -inf = 0
print(wei[0])

v = value(x)  # (B, T, 16): "if you attend to me, here's what I give you"

# Weighted sum: out[b,t] = Σ_j wei[b,t,j] * v[b,j]
# Each token aggregates information from past tokens, weighted by attention
out = wei @ v  # (B, T, T) @ (B, T, 16) → (B, T, 16)

out.shape  # torch.Size([4, 8, 16])




tensor([[-0.4407, -0.3253,  0.1413,  0.5404, -0.2668,  0.4908,  0.2691, -0.1132],
        [-0.8334, -0.4139,  0.0260,  0.8446, -0.5456,  0.2604, -0.0139,  0.0732],
        [-0.2557, -0.3152,  0.0191, -0.0953, -0.2461, -0.3576,  0.0187, -0.2387],
        [ 0.1959, -0.2004, -0.0842, -0.2124, -0.1401, -0.2925, -0.3232, -0.2565],
        [-0.3142,  0.0047, -0.1970, -0.3301,  0.5091,  0.2160,  0.0930,  0.2314],
        [-0.0782,  0.6038, -0.0276, -0.2483,  0.8362, -0.6307,  0.3547,  0.3049],
        [ 0.2719,  0.4913, -0.0655, -0.0789,  0.1523,  0.3154, -0.1371,  0.2012],
        [-0.4511, -0.1031, -0.2077,  0.1475, -0.1997, -0.1464,  0.1608,  0.1576]],
       grad_fn=<SelectBackward0>)
tensor([[-0.4407,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.8334, -0.4139,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.2557, -0.3152,  0.0191,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 0.1959, -0.2004, -0.0842, -0.2124,    -inf,    -inf, 

torch.Size([4, 8, 16])

In [32]:
wei[0]
# content which likely high affinity to each other

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3966, 0.6034, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3069, 0.2892, 0.4039, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3233, 0.2175, 0.2443, 0.2149, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1479, 0.2034, 0.1663, 0.1455, 0.3369, 0.0000, 0.0000, 0.0000],
        [0.1259, 0.2490, 0.1324, 0.1062, 0.3141, 0.0724, 0.0000, 0.0000],
        [0.1598, 0.1990, 0.1140, 0.1125, 0.1418, 0.1669, 0.1061, 0.0000],
        [0.0845, 0.1197, 0.1078, 0.1537, 0.1086, 0.1146, 0.1558, 0.1553]],
       grad_fn=<SelectBackward0>)

In [42]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5
wei_no_scaled = q @ k.transpose(-2,-1)


In [36]:
k.var()

tensor(0.9006)

In [37]:
q.var()

tensor(1.0037)

In [40]:
wei.var()

tensor(0.9957)

In [43]:
wei_no_scaled.var()

tensor(16.1036)

In [41]:
class LayerNorm1d: # (used to be BatchNorm1d)

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True) # batch mean
    xvar = x.var(1, keepdim=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [45]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('shaketext.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


0.209729 M parameters
step 0: train loss 4.4116, val loss 4.4022
step 100: train loss 2.6568, val loss 2.6670
step 200: train loss 2.5090, val loss 2.5058
step 300: train loss 2.4194, val loss 2.4334
step 400: train loss 2.3501, val loss 2.3568
step 500: train loss 2.2963, val loss 2.3129
step 600: train loss 2.2410, val loss 2.2501
step 700: train loss 2.2057, val loss 2.2191
step 800: train loss 2.1633, val loss 2.1860
step 900: train loss 2.1242, val loss 2.1498
step 1000: train loss 2.1027, val loss 2.1298
step 1100: train loss 2.0692, val loss 2.1183
step 1200: train loss 2.0386, val loss 2.0797
step 1300: train loss 2.0276, val loss 2.0652
step 1400: train loss 1.9925, val loss 2.0370
step 1500: train loss 1.9702, val loss 2.0302
step 1600: train loss 1.9645, val loss 2.0487
step 1700: train loss 1.9421, val loss 2.0143
step 1800: train loss 1.9091, val loss 1.9953
step 1900: train loss 1.9085, val loss 1.9874
step 2000: train loss 1.8861, val loss 1.9957
step 2100: train loss 1.